# Elderly Care Agentic Vision System — Interactive Demo
This notebook demonstrates the end-to-end AI + Vision pipeline for elderly indoor monitoring:
1. **Video Ingestion & Sampling**
2. **YOLOv8-Pose & Bed Overlap Estimation**
3. **Vision-Language Model (GPT-4o Vision / Offline Heuristics)**
4. **Temporal State Tracking & Debouncing**
5. **Agentic Disambiguation & Safety Alerts**
6. **Evaluation & Visualization**

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# Load environment variables
load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY", "mock")
print(f"API Key configured: {'Yes (Live GPT-4o)' if api_key != 'mock' else 'Running in Mock Mode'}")

## 1. Run the Pipeline
We can generate a synthetic demo video and run the entire pipeline in one call:

In [ ]:
from demo import create_synthetic_video
from pipeline import PipelineConfig, ElderlyCarePipeline

video_path = "demo_video.mp4"
create_synthetic_video(video_path, duration_sec=60, fps=10)

config = PipelineConfig(
    openai_api_key=api_key,
    vlm_model="gpt-4o",
    sample_interval_sec=2.0,
    output_dir="demo_output",
    enable_agentic_resolution=True,
    enable_pose_estimation=True,
    verbose=True,
)

pipeline = ElderlyCarePipeline(config)
report = pipeline.analyze(video_path)
print(f"\nAnalysis complete! Final alert: {report.final_alert.value}")

## 2. Visualize Activity Timeline and Durations

In [ ]:
with open("demo_output/activity_summary.json") as f:
    summary = json.load(f)

with open("demo_output/bed_events.json") as f:
    bed_events = json.load(f)

print("Activity Breakdown:")
for state, dur in summary["activity_summary"].items():
    print(f"  {state:<22} : {dur}")

print(f"\nBed Exits: {bed_events['bed_exit_count']} | Bed Returns: {bed_events['bed_return_count']}")
for ev in bed_events.get("events", []):
    print(f"  • [{ev['start_time']} -> {ev['confirmed_time']}] {ev['event'].upper()}: {ev['previous_state']} -> {ev['current_state']} (conf={ev['confidence']})")

## 3. Ground Truth Evaluation

In [ ]:
from evaluation import EvaluationMetrics

with open("ground_truth_template.json") as f:
    gt = json.load(f)

evaluator = EvaluationMetrics(report, gt)
metrics = evaluator.compute_all()
print(evaluator.format_report(metrics))